# Parte 1 · Entrenamiento de YOLOv12n para detectar entradas de OXXO
Entrena un detector de una clase (`entrada_oxxo`) a partir de `yolo12n.pt` con el dataset propio de Roboflow.

## 0. Preparación
Funciona en local o en Google Colab. En Colab: **Entorno de ejecución → Cambiar tipo → GPU** y corre la celda de clonado.

In [ ]:
# Solo en Colab: clona el repo y entra a la carpeta
import sys
if "google.colab" in sys.modules:
    !git clone https://github.com/DimoltoSs/oxxo-detection-yolo12.git
    %cd oxxo-detection-yolo12

In [ ]:
!pip install -q ultralytics==8.3.232

In [ ]:
from pathlib import Path

# Raíz del repo (sirve si el notebook se abre desde notebooks/ o desde la raíz)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET = ROOT / "dataset"
print("Repo:", ROOT)

## 1. Dataset
Descárgalo de [Roboflow Universe](https://universe.roboflow.com/angels-workspace-jvtq0/oxxo_detection/dataset/1) en formato **YOLOv12** y descomprime las carpetas `train/`, `valid/` y `test/` dentro de `dataset/` (el `data.yaml` del repo ya usa rutas relativas).

In [ ]:
for parte in ("train", "valid", "test"):
    carpeta = DATASET / parte / "images"
    n = len(list(carpeta.glob("*"))) if carpeta.exists() else 0
    print(f"{parte:5s}: {n} imágenes")
assert (DATASET / "train" / "images").exists(), "Falta el dataset: descárgalo de Roboflow (ver arriba)." 

## 2. Entrenamiento
100 épocas, imágenes de 640 px, lotes de 16. Usa la GPU si hay; si no, la CPU (mucho más lento).

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO("yolo12n.pt")   # pesos preentrenados (se descargan solos)
results = model.train(
    data=str(DATASET / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=16,
    workers=2,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(ROOT / "runs"),
    name="train",
)

## 3. Evaluación en el conjunto de prueba
El mejor modelo queda en `runs/train/weights/best.pt`. Cópialo a `weights/best.pt` si quieres reemplazar el del repo.

In [ ]:
best = YOLO(ROOT / "runs" / "train" / "weights" / "best.pt")
metricas = best.val(data=str(DATASET / "data.yaml"), split="test", imgsz=640)
print(f"mAP50: {metricas.box.map50:.3f} · mAP50-95: {metricas.box.map:.3f}")

In [ ]:
best.predict(
    source=str(DATASET / "test" / "images"),
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(ROOT / "predicciones"),
    name="test",
)